In [ ]:
!nvidia-smi

In [ ]:
!pip install ultralytics -q

In [ ]:
!pip install ultralytics roboflow -q

In [4]:
!pip install roboflow -q

In [ ]:
from roboflow import Roboflow

try:
    api_key = input("API Key: ")
    workspace = input("Workspace: ")
    project_name = input("Project Name: ")
    version_num = int(input("Version: "))

    rf = Roboflow(api_key=api_key)
    project = rf.workspace(workspace).project(project_name)
    version = project.version(version_num)
    dataset = version.download("yolov8")

    print("Done")
    print(f"Data path: {dataset.location}")

except ValueError:
    print("Error: Version must be a number")
except Exception as e:
    print("Error: Invalid input, check API Key, workspace, and project name")

In [ ]:
import os

print(" contant: ")
for item in os.listdir(dataset.location):
    print(f"  - {item}")

yaml_path = os.path.join(dataset.location, "data.yaml")
with open(yaml_path, "r") as f:
    print("\n data.yaml contant: ")
    print(f.read())

In [9]:
import IPython
display(IPython.display.Javascript('''
function ClickConnect(){
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)
'''))

<IPython.core.display.Javascript object>

In [ ]:
from ultralytics import YOLO
import os
model = YOLO("yolov10n.pt")

yaml_path = os.path.join(dataset.location, "data.yaml")

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=10,
    device=0,
    project="accident_detection",
    name="yolov8_run1",
    exist_ok=True
)

print("Done")

In [ ]:
from IPython.display import Image, display

results_path = "runs/detect/accident_detection/yolov8_run1/"

print("Result: ")
display(Image(filename=results_path + "results.png"))

print("Confusion Matrix:")
display(Image(filename=results_path + "confusion_matrix.png"))

In [ ]:
best_model = YOLO("runs/detect/accident_detection/yolov8_run1/weights/best.pt")

metrics = best_model.val()

print("result: ")
print(f"  mAP50:    {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")

In [ ]:
from google.colab import files
from IPython.display import Image, display
import glob

print("image testing")
uploaded = files.upload()

for filename in uploaded.keys():
    results = best_model.predict(
        source=filename,
        conf=0.25,
        save=True,
        project="test_results",
        name="prediction",
        exist_ok=True
    )


    output_images = glob.glob("test_results/prediction/*.jpg") + \
                    glob.glob("test_results/prediction/*.png")

    if output_images:
        print("\n Detection result:")
        display(Image(filename=output_images[0]))


In [ ]:
from google.colab import files

print(" Downloading best.pt ...")
files.download("runs/detect/accident_detection/yolov8_run1/weights/best.pt")
print("Done")